# Manipulación de datos previa al modelado

Toma la base final consolidada por el notebook de consignación (nivel predio/chip) y produce un dataset a nivel lote (`barmanpre`), agregando los chips que comparten lote:

- `dataset_ingenieria_caracteristicas.csv` → un registro por lote, `ph_bin` como control de propiedad horizontal (no se particiona en PH/NPH — unidad de analisis unica)

Decisiones metodológicas clave:
- Agregación por lote: valor y área construida se agregan como **promedio** (el lote representa el valor/área **típico de una unidad** del edificio, no el total construido).
- Área de terreno, distancias, coordenadas: **first** (compartidas por todos los chips del lote).
- Dummies de destino/uso/isócrona: **max** (si algún chip del lote tiene ese atributo, el lote lo hereda).
- `PreEstrato` ordinal, resto de categóricas como dummies (con umbral mínimo de categoría y agrupación de las raras en `Otros`).
- Transformaciones `log(x)` donde x > 0 siempre, `log(1+x)` donde x ≥ 0 — aplicadas **después** de agregar por lote.


### 1. Configuración e importación de librerías

Monta Google Drive (entorno Colab) e importa las librerías de manejo de datos tabulares (`pandas`), numéricos (`numpy`) y geoespaciales (`geopandas`, `shapely`). Define la ruta de entrada (`PATH_IN`) hacia el dataset consolidado que produce el notebook de consignación, y fija el CRS objetivo (`EPSG:6247`, sistema de referencia oficial para Bogotá) que se usará al reconstruir la geometría.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import warnings

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

PATH_IN    = '/content/drive/MyDrive/Catastro/dataset_final_predios_completo.csv'
CRS_TARGET = 'EPSG:6247'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Carga de datos y reconstrucción de la geometría

Carga el CSV consolidado a nivel predio/chip. El identificador `barmanpre` se fuerza a texto y se rellena con ceros a la izquierda (`zfill`) para evitar que pandas lo interprete como numérico y pierda ceros iniciales, ya que es la llave que se usará más adelante para agregar a nivel lote.

Como el CSV no puede guardar geometrías nativas, la columna `geometry` llega como texto en formato WKT; aquí se reconstruye con `shapely.wkt.loads` y se arma el `GeoDataFrame` (`gdf`) con el CRS objetivo. Si el CSV también trae la columna `centroide` en WKT, se reconstruye de la misma forma. De aquí en adelante todas las celdas trabajan sobre `gdf`, no sobre `df`.


In [ ]:
# ── 1. CARGA ────────────────────────────────────────────────────────────
df = pd.read_csv(PATH_IN, encoding='utf-8-sig', dtype={'barmanpre': str})

df['barmanpre'] = df['barmanpre'].str.zfill(12)

print(f'Barmanpre - largos: {df["barmanpre"].str.len().value_counts().to_dict()}')
print(df['barmanpre'].head(10).tolist())

# Reconstruir geometria desde WKT (el CSV la trae como texto) y convertir
# a GeoDataFrame, ya que las celdas siguientes usan 'gdf', no 'df'
df['geometry'] = df['geometry'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=CRS_TARGET)

# Si el CSV tambien trae 'centroide' como texto, reconstruirlo igual
if 'centroide' in gdf.columns:
    gdf['centroide'] = gdf['centroide'].apply(wkt.loads)

print(f'\nGeoDataFrame construido: {len(gdf):,} filas')
print(f'CRS: {gdf.crs}')

Barmanpre - largos: {12: 103110}
['008205049002', '008205049002', '008205049002', '008205049002', '008205049002', '008205049002', '008205049002', '008205049002', '008205049002', '008205049005']

GeoDataFrame construido: 103,110 filas
CRS: EPSG:6247


### 3. Filtrado de registros inválidos

Revisa nulos en las variables críticas para el modelo (avalúo, áreas, estrato, destino y uso) y descarta los chips cuyo `valor_avaluo_2026` o `area_terreno_2026` sean nulos o menores o iguales a cero, ya que estos casos no pueden entrar en las transformaciones logarítmicas posteriores. Se reporta cuántos chips quedan excluidos y cuántos lotes distintos siguen representados tras el filtro.


In [ ]:
cols_criticas = ['valor_avaluo_2026', 'area_terreno_2026', 'area_construida_2026',
                 'PreEstrato', 'PreDestino_desc', 'PreUsoPH']

print('Nulos por variable clave:')
print(gdf[cols_criticas].isna().sum())

n_antes = len(gdf)
gdf = gdf[gdf['valor_avaluo_2026'].notna() & (gdf['valor_avaluo_2026'] > 0)].copy()
gdf = gdf[gdf['area_terreno_2026'].notna() & (gdf['area_terreno_2026'] > 0)].copy()

print(f'\nExcluidos por avaluo/area invalidos: {n_antes - len(gdf):,}')
print(f'Chips validos: {len(gdf):,}')
print(f'Lotes representados: {gdf["barmanpre"].nunique():,}')

Nulos por variable clave:
valor_avaluo_2026         0
area_terreno_2026         0
area_construida_2026     14
PreEstrato                0
PreDestino_desc           0
PreUsoPH                349
dtype: int64

Excluidos por avaluo/area invalidos: 32
Chips validos: 103,078
Lotes representados: 16,613


### 4. Transformación logarítmica de distancias (nivel chip)

Aplica `log1p` (log(1+x), válido para x ≥ 0) a las distancias euclidianas hacia metro, viaducto, Transmilenio, parques, salud y educación. Estas transformaciones se calculan todavía a nivel chip, antes de agregar por lote, porque las distancias son atributos que se heredan igual para todos los chips de un mismo lote (se toman por `first` más adelante).


In [ ]:
# ── DISTANCIAS LOG-TRANSFORMADAS (nivel chip) ──────────────────────────────
for col in ['dist_estacion_metro_m', 'dist_viaducto_m', 'dist_tm_m',
            'dist_parques_m', 'dist_salud_m', 'dist_educacion_m']:
    gdf[f'log_{col}'] = np.log1p(gdf[col])

print('Distancias log-transformadas (nivel chip):')
cols_log_dist = [f'log_{c}' for c in ['dist_estacion_metro_m','dist_viaducto_m','dist_tm_m',
                                       'dist_parques_m','dist_salud_m','dist_educacion_m']]
print(gdf[cols_log_dist].describe().T[['mean','std','min','max']])


Distancias log-transformadas (nivel chip):
                            mean    std    min    max
log_dist_estacion_metro_m 6.8326 0.5082 3.4780 7.5958
log_dist_viaducto_m       6.0542 0.8420 3.0312 7.2342
log_dist_tm_m             6.3567 0.6348 3.3497 7.4869
log_dist_parques_m        4.6605 0.8427 0.0000 6.5342
log_dist_salud_m          4.6882 1.1331 0.0000 6.6374
log_dist_educacion_m      5.3308 0.6824 2.0082 6.8139


### 5. Variables de estrato y propiedad horizontal (PH)

Convierte `PreEstrato` a numérico (`errors='coerce'` para que valores no parseables queden como nulo en vez de romper el pipeline) y construye la bandera binaria `es_PH`, que identifica si un chip corresponde a una unidad bajo régimen de propiedad horizontal (`PreUsoPH == 'S'`). Se imprime la distribución de estrato y la proporción de chips PH vs. no PH como chequeo rápido antes de pasar a la agregación por lote.


In [ ]:
gdf['PreEstrato'] = pd.to_numeric(gdf['PreEstrato'], errors='coerce')

gdf['es_PH'] = (
    gdf['PreUsoPH'].astype(str).str.upper() == 'S'
).astype(int)

print('Distribucion PreEstrato (a nivel chip):')
print(gdf['PreEstrato'].value_counts(dropna=False).sort_index())
print(f'\nDistribucion PH/NPH:')
print(f'  Chips PH:  {gdf["es_PH"].sum():,}  ({gdf["es_PH"].mean()*100:.1f}%)')
print(f'  Chips NPH: {(1-gdf["es_PH"]).sum():,}  ({(1-gdf["es_PH"]).mean()*100:.1f}%)')

Distribucion PreEstrato (a nivel chip):
PreEstrato
0    34383
2      223
3    20713
4    32701
5     9035
6     6023
Name: count, dtype: int64

Distribucion PH/NPH:
  Chips PH:  97,991  (95.1%)
  Chips NPH: 5,087  (4.9%)


### 6. `procesar_umbral`: función central de agregación chip → lote

Esta es la celda más pesada del notebook: define la función que transforma el dataset de nivel **chip** a nivel **lote** (`barmanpre`), aplicando el criterio de agregación descrito en la celda de introducción. El parámetro `umbral` controla cuántas observaciones mínimas necesita una categoría de `PreDestino_desc` para no ser agrupada en `'Otros'`.

Pasos que ejecuta, en orden:

1. **Agrupación de categorías raras**: categorías de `PreDestino_desc` con menos de `umbral` observaciones se reemplazan por `'Otros'`.
2. **Destino y uso por moda de lote (corrección de bug)**: `PreDestino_desc` y `PreUso_desc` se recalculan como la **moda** entre los chips de cada lote — no se asigna el valor con que "al menos un chip" del lote venía etiquetado. Esto corrige el error de agregación por mayoría detectado previamente sobre estas dos variables (ver `es_PH` más abajo, mismo criterio). Se reporta cuántos chips cambiaron de categoría al aplicar la moda, como evidencia de que la corrección tuvo efecto real sobre los datos.
3. **Dummies categóricas**: `PreDestino_desc`, `PreUso_desc`, `banda_iso_m` y `estacion_iso` se convierten a variables dummy, fijando `'Residencial'` como categoría de referencia (en vez de dejar que `drop_first` elimine la primera categoría en orden alfabético, lo cual haría que la referencia cambiara arbitrariamente según los datos).
4. **Agregación por lote** (`groupby('barmanpre')`), con una regla distinta según el tipo de variable:
   - `mean` para `valor_avaluo_2026` y `area_construida_2026` (el lote representa el valor/área **típico de una unidad**, no el total construido).
   - `first` para área de terreno, distancias log-transformadas, coordenadas y variables de proximidad (son iguales para todos los chips de un mismo lote).
   - `max` para las dummies de destino/uso/isócrona (si algún chip del lote tiene el atributo, el lote lo hereda).
   - `mean` seguido de umbral en 0.5 para `es_PH` (mayoría simple: el lote es PH si la mayoría de sus chips lo son).
5. **Limpieza post-agregación**: dummies que, ya a nivel lote, tienen menos de `min_casos` lotes activos se descartan (evita variables con muy poca variación que solo añadirían ruido al modelo).
6. **Transformaciones finales**: `log_valor`, `log_area_terreno`, `log_area_construida`, `valor_m2` y `log_valor_m2` se calculan **después** de agregar por lote, sobre los promedios ya agregados — no antes.

La función auxiliar `preparar_export` al final simplemente convierte las columnas de geometría a WKT (texto) para que el resultado se pueda exportar a CSV sin perder la información espacial.


In [ ]:
def procesar_umbral(gdf_chip, umbral, min_casos=10, verbose=True):
    gdf_u = gdf_chip.copy()  # copia fresca: evita contaminar entre iteraciones del loop

    # -- Agrupar categorias pequeñas de PreDestino_desc en 'Otros' --
    conteo = gdf_u['PreDestino_desc'].value_counts()
    categorias_raras = conteo[conteo < umbral].index.tolist()
    gdf_u['PreDestino_desc'] = gdf_u['PreDestino_desc'].replace(
        {cat: 'Otros' for cat in categorias_raras}
    )
    if verbose:
        print(f'[umbral={umbral}] Categorias agrupadas en Otros ({len(categorias_raras)}): {categorias_raras}')

    # ── CAMBIO 1: destino por lote = MODA entre sus chips, no "al menos un chip" ──
    destino_por_lote = (
        gdf_u.groupby('barmanpre')['PreDestino_desc']
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else None)
        .rename('PreDestino_desc_lote')
    )
    gdf_u = gdf_u.merge(destino_por_lote, on='barmanpre', how='left')
    n_cambiados_dest = (gdf_u['PreDestino_desc'] != gdf_u['PreDestino_desc_lote']).sum()
    gdf_u['PreDestino_desc'] = gdf_u['PreDestino_desc_lote']
    gdf_u = gdf_u.drop(columns=['PreDestino_desc_lote'])
    if verbose:
        print(f'[umbral={umbral}] Chips cuyo destino cambió al usar moda del lote: {n_cambiados_dest:,}')

    # ── CAMBIO 2: uso por lote = MODA entre sus chips, mismo criterio ──
    uso_por_lote = (
        gdf_u.groupby('barmanpre')['PreUso_desc']
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else None)
        .rename('PreUso_desc_lote')
    )
    gdf_u = gdf_u.merge(uso_por_lote, on='barmanpre', how='left')
    n_cambiados_uso = (gdf_u['PreUso_desc'] != gdf_u['PreUso_desc_lote']).sum()
    gdf_u['PreUso_desc'] = gdf_u['PreUso_desc_lote']
    gdf_u = gdf_u.drop(columns=['PreUso_desc_lote'])
    if verbose:
        print(f'[umbral={umbral}] Chips cuyo uso cambió al usar moda del lote: {n_cambiados_uso:,}')

    # -- Dummies, con Residencial fijo como referencia (no dropea alfabetico) --
    cats_dummies = ['PreDestino_desc', 'PreUso_desc', 'banda_iso_m', 'estacion_iso']
    gdf_u['PreDestino_desc'] = pd.Categorical(
        gdf_u['PreDestino_desc'],
        categories=['Residencial'] + [c for c in gdf_u['PreDestino_desc'].unique() if c != 'Residencial']
    )
    gdf_dum_u = pd.get_dummies(
        gdf_u, columns=cats_dummies, prefix=['dest', 'uso', 'iso', 'est_iso'],
        drop_first=True, dummy_na=False,
    )
    bool_cols = gdf_dum_u.select_dtypes(include='bool').columns
    gdf_dum_u[bool_cols] = gdf_dum_u[bool_cols].astype(int)

    # -- Agregacion por lote (barmanpre) --
    cols_promedio = ['valor_avaluo_2026', 'area_construida_2026']
    cols_first = [
        'area_terreno_2026', 'PreEstrato',
        'log_dist_estacion_metro_m', 'log_dist_viaducto_m', 'log_dist_tm_m',
        'log_dist_parques_m', 'log_dist_salud_m', 'log_dist_educacion_m',
        'cx', 'cy', 'estacion_metro_cercana', 'estacion_tm_cercana',
    ]
    cols_dummies = [c for c in gdf_dum_u.columns if any(
        c.startswith(p) for p in ['dest_', 'uso_', 'iso_', 'est_iso_']
    )]
    agg_dict = {}
    agg_dict.update({c: 'mean'  for c in cols_promedio if c in gdf_dum_u.columns})
    agg_dict.update({c: 'first' for c in cols_first    if c in gdf_dum_u.columns})
    agg_dict.update({c: 'max'   for c in cols_dummies  if c in gdf_dum_u.columns})

    agg_dict['es_PH'] = 'mean'

    gdf_lote_u = gdf_dum_u.groupby('barmanpre', as_index=False).agg(agg_dict)
    gdf_lote_u['es_PH'] = (gdf_lote_u['es_PH'] >= 0.5).astype(int)
    n_chips = gdf_dum_u.groupby('barmanpre').size().rename('n_chips_por_lote')
    gdf_lote_u = gdf_lote_u.merge(n_chips, on='barmanpre', how='left')

    # -- Limpieza de dummies raras post-agregacion --
    a_botar = [c for c in gdf_lote_u.columns
               if any(c.startswith(p) for p in ('dest_','uso_','iso_','est_iso_'))
               and gdf_lote_u[c].sum() < min_casos]
    if a_botar and verbose:
        print(f'[umbral={umbral}] Dummies descartadas (n < {min_casos} lotes): {a_botar}')
    gdf_lote_u = gdf_lote_u.drop(columns=a_botar)

    # -- Logs de valor/area (post-agregacion, sobre el promedio) --
    gdf_lote_u['log_valor']           = np.log(gdf_lote_u['valor_avaluo_2026'])
    gdf_lote_u['log_area_terreno']    = np.log(gdf_lote_u['area_terreno_2026'])
    gdf_lote_u['log_area_construida'] = np.log1p(gdf_lote_u['area_construida_2026'])
    gdf_lote_u['valor_m2'] = np.where(
        gdf_lote_u['area_construida_2026'] > 0,
        gdf_lote_u['valor_avaluo_2026'] / gdf_lote_u['area_construida_2026'],
        np.nan
    )
    gdf_lote_u['log_valor_m2']   = np.log(gdf_lote_u['valor_m2'])
    gdf_lote_u['ph_bin']         = gdf_lote_u['es_PH']
    gdf_lote_u['log_valor_avaluo'] = gdf_lote_u['log_valor']

    if verbose:
        print(f'[umbral={umbral}] Chips: {len(gdf_dum_u):,} -> Lotes: {len(gdf_lote_u):,}  '
              f'| Columnas: {len(gdf_lote_u.columns)}  '
              f'| dest_ dummies: {sum(c.startswith("dest_") for c in gdf_lote_u.columns)}')

    return gdf_lote_u


def preparar_export(df):
    out = df.copy()
    if 'geometry' in out.columns:
        out['geometry'] = out.geometry.to_wkt()
    if 'centroide' in out.columns:
        out['centroide'] = out['centroide'].astype(str)
    return pd.DataFrame(out)

### 7. Ejecución del pipeline y exportación

Corre `procesar_umbral` sobre `gdf` y exporta el resultado como `dataset_ingenieria_caracteristicas.csv`, el archivo que consume directamente el notebook `Modelos_de_valor`.


In [ ]:
# ── PROCESAR Y EXPORTAR ─────────────────────────────────────────────────
gdf_lote = procesar_umbral(gdf, 100, min_casos=10)

path_out = '/content/drive/MyDrive/Catastro/dataset_ingenieria_caracteristicas.csv'
preparar_export(gdf_lote).to_csv(path_out, index=False, encoding='utf-8-sig')

print(f'Exportado: {path_out}  ({len(gdf_lote):,} lotes)')


### 8. Diagnóstico y validación del dataset de lotes

Corre un chequeo de calidad sobre el dataset final a nivel lote: conteo de lotes, media/desviación de `log_valor`, distribución de estrato y de `ph_bin`, cobertura dentro de bandas de isócrona, y nulos en las variables que entrarán al modelo. Cierra con la matriz de correlaciones entre las variables continuas y la distribución de las dummies `dest_` más frecuentes, como última verificación antes de dar por cerrado este dataset para modelado.


In [ ]:
# ── DIAGNOSTICO / RESUMEN
def resumen(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Lotes: {len(df):,}')
    print(f'log_valor  -> media: {df["log_valor"].mean():.3f}   std: {df["log_valor"].std():.3f}')
    print(f'PreEstrato distribucion: {dict(df["PreEstrato"].value_counts(dropna=False).sort_index())}')
    print(f'ph_bin distribucion: {dict(df["ph_bin"].value_counts(dropna=False).sort_index())}')

    dummies_iso = [c for c in df.columns if c.startswith('iso_')]
    dentro = df[dummies_iso].sum(axis=1) > 0 if dummies_iso else pd.Series([False]*len(df))
    print(f'Lotes dentro de alguna banda de isocrona: {dentro.sum():,} ({dentro.mean()*100:.1f}%)')

    cols_modelo = ['log_valor','log_area_terreno','log_area_construida',
                   'log_dist_estacion_metro_m','log_dist_viaducto_m',
                   'log_dist_parques_m','log_dist_salud_m','log_dist_educacion_m',
                   'PreEstrato']
    nulos = df[cols_modelo].isna().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print('Nulos en variables de modelo:')
        print(nulos)
    else:
        print('Nulos en variables de modelo: ninguno')

resumen(gdf_lote, 'Lotes')

cols_continuas = ['log_valor','log_area_terreno','log_area_construida',
                  'log_dist_estacion_metro_m','log_dist_viaducto_m','log_dist_tm_m',
                  'log_dist_parques_m','log_dist_salud_m','log_dist_educacion_m',
                  'PreEstrato']
print('\n=== Correlaciones (nivel lote) ===')
print(gdf_lote[cols_continuas].corr().round(2))

print('\nDistribución dest_ (top 5):')
dest_cols = [c for c in df.columns if c.startswith('dest_')]
print(df[dest_cols].sum().sort_values(ascending=False).head(5))
